# Strategy

This notebook builds a **CatBoost multiclass classifier** and focuses on a practical balance of:

- feature engineering
- leakage-safe target encoding
- ORIG-based prior features
- post-hoc threshold optimization for **balanced accuracy**

---

## Core idea

### 1. Use ORIG as a feature source, not as additional training rows
The original dataset is used to construct informative priors such as:

- category-level target statistics
- frequency information
- distribution-based numeric references
- conditional median references

However, **ORIG rows themselves are not appended to the training data** in this notebook.  
This keeps the training distribution aligned with the competition data while still leveraging the structure of the original data.

---

### 2. Practical nested CV
The training pipeline uses:

- **Outer 10-fold CV**
- **Inner 5-fold CV**

Heavy feature engineering is computed **once outside CV** whenever safe, while **OOF target encoding** is generated inside the fold structure to reduce leakage.

This gives a practical compromise between:
- strict leakage control
- manageable runtime
- stable OOF estimates

---

## Feature engineering design

### Base features
We start from lightweight but expressive tabular features:

- arithmetic interactions
- ratio features
- threshold-based rule features
- decimal digit features
- log-transformed features
- quantile-bin features
- categorical cross features

### ORIG-based features
From the original dataset, we derive:

- `orig_mean`
- `orig_smooth`
- `orig_woe`
- `orig_entropy`

for categorical variables.

### Distribution-aware numeric features
Using ORIG as the numeric reference distribution, we add:

- distribution position
- z-like relative position
- q50 distance features
- conditional q50 distance features

This helps the model capture not only raw values, but also **where a value sits relative to the original data distribution**.

### Frequency features
Frequency encodings are computed using train/test/orig references for selected categorical-style features.

---

## Target encoding policy
Target encoding is handled with **OOF construction** inside the CV pipeline.

We do not fit TE on the full training fold and reuse it blindly on the same data.  
Instead, TE features for the train side are created through inner-fold OOF logic, while validation/test receive mappings built from the corresponding training subset.

This is a practical leakage-safe approach.

---

## Metric alignment
The competition metric is **balanced accuracy**.

Because the model is trained with a probabilistic multiclass objective, the raw `argmax` prediction is not always optimal for balanced accuracy.  
Therefore, after OOF predictions are obtained, we perform **threshold / bias optimization** on OOF probabilities.

We compare:

- baseline argmax
- class-wise probability weighting
- class-wise log-bias adjustment

and apply the best strategy to the test predictions.

---

## What this notebook aims to achieve
This notebook is designed to be:

- strong enough for serious leaderboard use
- interpretable in terms of feature strategy
- safe from major leakage issues
- easy to extend into ensembles / stacking later

In short, the pipeline is:

**ORIG-informed features + practical OOF TE + CatBoost + threshold optimization**

---

## Pipeline summary
1. Load competition train/test and original dataset
2. Build base handcrafted features
3. Build ORIG-derived priors and distribution features
4. Add leakage-safe OOF target encoding
5. Train CatBoost with outer 10-fold / inner 5-fold CV
6. Collect OOF and test probabilities
7. Optimize final class decision thresholds on OOF
8. Generate final submission

In [1]:
import os
import gc
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score


# =========================================================
# Config
# =========================================================
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
ORIG_PATH  = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

ID_COL = "id"
TARGET = "Irrigation_Need"

N_OUTER = 10
N_INNER = 5
SEED = 202

USE_ORIG_AS_REFERENCE = True
THRESH_MAXITER = 300
EPS = 1e-15

CAT_PARAMS = {
    "loss_function": "MultiClass",
    "eval_metric": "TotalF1",
    "iterations": 5000,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 5.0,
    "random_strength": 1.0,
    "bagging_temperature": 0.5,
    "grow_policy": "SymmetricTree",
    "random_seed": SEED,
    "verbose": 200,
    "allow_writing_files": False,
    "early_stopping_rounds": 200,
    "task_type": "GPU",
    "devices":'0'
}


# =========================================================
# Seed
# =========================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

In [2]:
# =========================================================
# Base FE + Domain FE
# =========================================================
def add_basic_features(df):
    df = df.copy()
    eps = 1e-6

    # -----------------------------------------------------
    # Basic arithmetic features
    # -----------------------------------------------------
    if {"Temperature_C", "Humidity"}.issubset(df.columns):
        df["Temp_x_Humidity"] = df["Temperature_C"] * df["Humidity"]
        df["Temp_div_Humidity"] = df["Temperature_C"] / (df["Humidity"] + 1.0)
        df["Humidity_div_Temp"] = df["Humidity"] / (df["Temperature_C"] + 1.0)

    if {"Rainfall_mm", "Previous_Irrigation_mm"}.issubset(df.columns):
        df["Total_Water_Input"] = df["Rainfall_mm"] + df["Previous_Irrigation_mm"]
        df["Rain_minus_PrevIrr"] = df["Rainfall_mm"] - df["Previous_Irrigation_mm"]
        df["Rain_div_PrevIrr"] = df["Rainfall_mm"] / (df["Previous_Irrigation_mm"] + 1.0)
        df["PrevIrr_div_Rain"] = df["Previous_Irrigation_mm"] / (df["Rainfall_mm"] + 1.0)
        df["Irrigation_Ratio"] = df["Previous_Irrigation_mm"] / (df["Rainfall_mm"] + 1.0)

    if {"Field_Area_hectare", "Previous_Irrigation_mm"}.issubset(df.columns):
        df["Area_x_PrevIrr"] = df["Field_Area_hectare"] * df["Previous_Irrigation_mm"]
        df["Irrig_Per_Ha"] = df["Previous_Irrigation_mm"] / (df["Field_Area_hectare"] + 0.1)

    if {"Field_Area_hectare", "Rainfall_mm"}.issubset(df.columns):
        df["Rain_Per_Ha"] = df["Rainfall_mm"] / (df["Field_Area_hectare"] + 0.1)

    if {"Field_Area_hectare", "Total_Water_Input"}.issubset(df.columns):
        df["Water_Per_Ha"] = df["Total_Water_Input"] / (df["Field_Area_hectare"] + 0.1)

    if {"Soil_Moisture", "Organic_Carbon"}.issubset(df.columns):
        df["Moisture_Retention"] = df["Soil_Moisture"] * df["Organic_Carbon"]

    if {"Electrical_Conductivity", "Soil_pH"}.issubset(df.columns):
        df["EC_pH_Interaction"] = df["Electrical_Conductivity"] * df["Soil_pH"]

    if {"Sunlight_Hours", "Temperature_C"}.issubset(df.columns):
        df["Sunlight_x_Temp"] = df["Sunlight_Hours"] * df["Temperature_C"]

    if {"Wind_Speed_kmh", "Humidity"}.issubset(df.columns):
        df["Wind_x_Humidity"] = df["Wind_Speed_kmh"] * df["Humidity"]

    # -----------------------------------------------------
    # Rule / threshold features
    # -----------------------------------------------------
    SOIL_THRESH = 25
    RAIN_THRESH = 300
    TEMP_THRESH = 30
    WIND_THRESH = 10

    if "Soil_Moisture" in df.columns:
        df["soil_lt_25"] = (df["Soil_Moisture"] < SOIL_THRESH).astype(int)
        df["soil_dist_25"] = df["Soil_Moisture"] - SOIL_THRESH

    if "Rainfall_mm" in df.columns:
        df["rain_lt_300"] = (df["Rainfall_mm"] < RAIN_THRESH).astype(int)
        df["rain_dist_300"] = df["Rainfall_mm"] - RAIN_THRESH

    if "Temperature_C" in df.columns:
        df["temp_gt_30"] = (df["Temperature_C"] > TEMP_THRESH).astype(int)
        df["temp_dist_30"] = df["Temperature_C"] - TEMP_THRESH

    if "Wind_Speed_kmh" in df.columns:
        df["wind_gt_10"] = (df["Wind_Speed_kmh"] > WIND_THRESH).astype(int)
        df["wind_dist_10"] = df["Wind_Speed_kmh"] - WIND_THRESH

    if "Crop_Growth_Stage" in df.columns:
        df["is_harvest"] = (df["Crop_Growth_Stage"] == "Harvest").astype(int)
        df["is_sowing"] = (df["Crop_Growth_Stage"] == "Sowing").astype(int)

    if "Mulching_Used" in df.columns:
        df["Mulch_Flag"] = df["Mulching_Used"].map({"Yes": 1.0, "No": 0.0}).fillna(0.0).astype(np.float32)

    if {"soil_lt_25", "rain_lt_300", "temp_gt_30", "wind_gt_10", "is_harvest", "is_sowing", "Mulch_Flag"}.issubset(df.columns):
        high_score = 2 * df["soil_lt_25"] + 2 * df["rain_lt_300"] + df["temp_gt_30"] + df["wind_gt_10"]
        low_score = 2 * df["is_harvest"] + 2 * df["is_sowing"] + df["Mulch_Flag"]
        df["magic_score"] = high_score - low_score
        df["dist_boundary_0"] = np.abs(df["magic_score"] - 0)
        df["dist_boundary_3"] = np.abs(df["magic_score"] - 3)
        df["dist_boundary_min"] = np.minimum(df["dist_boundary_0"], df["dist_boundary_3"])


    # -----------------------------------------------------
    # Digit features
    # -----------------------------------------------------
    digit_cols = [
        "Soil_Moisture", "Temperature_C", "Rainfall_mm", "Wind_Speed_kmh",
        "Humidity", "Soil_pH", "Organic_Carbon", "Electrical_Conductivity",
        "Sunlight_Hours", "Field_Area_hectare", "Previous_Irrigation_mm"
    ]
    for col in digit_cols:
        if col in df.columns:
            vals = df[col].values
            df[f"{col}_dec"] = np.floor((vals - np.floor(vals)) * 10).astype(int)

    # -----------------------------------------------------
    # Log features
    # -----------------------------------------------------
    for col in ["Rainfall_mm", "Previous_Irrigation_mm", "Field_Area_hectare", "Wind_Speed_kmh"]:
        if col in df.columns:
            df[f"log_{col}"] = np.log1p(df[col])

    # -----------------------------------------------------
    # Bin / round features
    # -----------------------------------------------------
    num_bin_cols = [
        "Soil_pH", "Soil_Moisture", "Temperature_C", "Humidity",
        "Rainfall_mm", "Sunlight_Hours", "Wind_Speed_kmh",
        "Field_Area_hectare", "Previous_Irrigation_mm",
        "Organic_Carbon", "Electrical_Conductivity"
    ]
    for col in num_bin_cols:
        if col not in df.columns:
            continue
        try:
            df[f"{col}_qbin10"] = pd.qcut(df[col], q=10, labels=False, duplicates="drop").astype("Int64").astype(str)
        except Exception:
            pass
        try:
            df[f"{col}_round1"] = df[col].round(1).astype(str)
        except Exception:
            pass

    # -----------------------------------------------------
    # Cross categorical features
    # -----------------------------------------------------
    cross_pairs = [
        ("Soil_Type", "Crop_Type"),
        ("Crop_Type", "Season"),
        ("Irrigation_Type", "Water_Source"),
        ("Region", "Season"),
        ("Crop_Growth_Stage", "Season"),
        ("Mulching_Used", "Crop_Type"),
        ("Soil_Type", "Season"),
        ("Region", "Crop_Type"),
    ]
    for c1, c2 in cross_pairs:
        if c1 in df.columns and c2 in df.columns:
            df[f"{c1}_x_{c2}"] = df[c1].astype(str) + "__" + df[c2].astype(str)

    return df


# =========================================================
# ORIG stats
# =========================================================
def build_orig_stats_maps(orig_df, cols, target_col, class_names, smoothing=20):
    stats_maps = {}
    n_total = len(orig_df)
    priors = {cls: (orig_df[target_col] == cls).mean() for cls in class_names}

    for col in cols:
        tmp = orig_df[[col, target_col]].copy()
        g = tmp.groupby(col, dropna=False)

        count = g.size().rename("count")
        merged = pd.DataFrame(count)

        for cls in class_names:
            cls_count = g[target_col].apply(lambda s: (s == cls).sum()).rename(f"cnt_{cls}")
            merged = merged.join(cls_count)

        col_maps = {}
        for cls in class_names:
            col_maps[f"{col}_orig_mean_{cls}"] = {}
            col_maps[f"{col}_orig_smooth_{cls}"] = {}
            col_maps[f"{col}_orig_woe_{cls}"] = {}
        col_maps[f"{col}_orig_entropy"] = {}

        for key in merged.index.tolist():
            row = merged.loc[key]
            cnt = row["count"]
            probs = []

            for cls in class_names:
                p = row[f"cnt_{cls}"] / cnt if cnt > 0 else 0.0
                probs.append(p)

                raw_mean = p
                smooth_mean = (row[f"cnt_{cls}"] + smoothing * priors[cls]) / (cnt + smoothing)

                pos = row[f"cnt_{cls}"]
                neg = cnt - pos
                total_pos = (orig_df[target_col] == cls).sum()
                total_neg = n_total - total_pos

                rate_pos = (pos + 0.5) / (total_pos + 1.0)
                rate_neg = (neg + 0.5) / (total_neg + 1.0)
                woe = np.log(rate_pos / rate_neg)

                col_maps[f"{col}_orig_mean_{cls}"][key] = float(raw_mean)
                col_maps[f"{col}_orig_smooth_{cls}"][key] = float(smooth_mean)
                col_maps[f"{col}_orig_woe_{cls}"][key] = float(woe)

            probs = np.clip(np.array(probs, dtype=float), 1e-12, 1.0)
            ent = -np.sum(probs * np.log(probs))
            col_maps[f"{col}_orig_entropy"][key] = float(ent)

        defaults = {}
        for cls in class_names:
            defaults[f"{col}_orig_mean_{cls}"] = float(priors[cls])
            defaults[f"{col}_orig_smooth_{cls}"] = float(priors[cls])

            total_pos = (orig_df[target_col] == cls).sum()
            total_neg = n_total - total_pos
            defaults[f"{col}_orig_woe_{cls}"] = float(
                np.log(((0.5) / (total_pos + 1.0)) / ((0.5) / (total_neg + 1.0)))
            )

        class_prior_arr = np.array([priors[c] for c in class_names], dtype=float)
        class_prior_arr = np.clip(class_prior_arr, 1e-12, 1.0)
        defaults[f"{col}_orig_entropy"] = float(-np.sum(class_prior_arr * np.log(class_prior_arr)))

        stats_maps[col] = {"maps": col_maps, "defaults": defaults}

    return stats_maps


def apply_orig_stats_features(df, stats_maps, cols):
    df = df.copy()
    for col in cols:
        if col not in stats_maps:
            continue
        maps_dict = stats_maps[col]["maps"]
        defaults = stats_maps[col]["defaults"]
        for feat_name, mp in maps_dict.items():
            df[feat_name] = df[col].map(mp).fillna(defaults[feat_name]).astype(np.float32)
    return df


# =========================================================
# Distribution / quantile / conditional q50
# =========================================================
def build_distribution_maps_from_reference(ref_df, num_cols, quantiles=(0.5,)):
    maps = {}
    for col in num_cols:
        arr = pd.to_numeric(ref_df[col], errors="coerce").dropna().values.astype(float)

        if len(arr) == 0:
            maps[col] = {
                "sorted": np.array([], dtype=float),
                "quantiles": {q: 0.0 for q in quantiles},
                "mean": 0.0,
                "std": 1.0,
                "median": 0.0,
            }
            continue

        std_ = float(np.std(arr))
        if std_ < 1e-8:
            std_ = 1.0

        maps[col] = {
            "sorted": np.sort(arr),
            "quantiles": {q: float(np.quantile(arr, q)) for q in quantiles},
            "mean": float(np.mean(arr)),
            "std": std_,
            "median": float(np.median(arr)),
        }
    return maps


def apply_distribution_position_features(df, dist_maps, num_cols):
    df = df.copy()
    for col in num_cols:
        vals = pd.to_numeric(df[col], errors="coerce").values.astype(float)
        sorted_arr = dist_maps[col]["sorted"]
        mean_ = dist_maps[col]["mean"]
        std_ = dist_maps[col]["std"]

        if len(sorted_arr) == 0:
            df[f"{col}_dist_pos"] = np.float32(0.5)
            df[f"{col}_dist_z"] = np.float32(0.0)
            continue

        pos = np.searchsorted(sorted_arr, vals, side="right") / len(sorted_arr)
        pos = np.where(np.isnan(vals), 0.5, pos)
        z = (vals - mean_) / std_
        z = np.where(np.isnan(vals), 0.0, z)

        df[f"{col}_dist_pos"] = pos.astype(np.float32)
        df[f"{col}_dist_z"] = z.astype(np.float32)
    return df


def apply_quantile_distance_features(df, dist_maps, num_cols, quantiles=(0.5,)):
    df = df.copy()
    for col in num_cols:
        vals = pd.to_numeric(df[col], errors="coerce").values.astype(float)
        fill_base = dist_maps[col]["median"]
        vals = np.where(np.isnan(vals), fill_base, vals)

        for q in quantiles:
            qv = dist_maps[col]["quantiles"][q]
            qname = int(q * 100)
            df[f"{col}_q{qname}_diff"] = (vals - qv).astype(np.float32)
            df[f"{col}_q{qname}_absdiff"] = np.abs(vals - qv).astype(np.float32)
    return df


def build_conditional_median_maps_from_reference(ref_df, num_cols, cond_cat_cols):
    cond_maps = {}
    global_medians = {
        num_col: float(pd.to_numeric(ref_df[num_col], errors="coerce").median())
        for num_col in num_cols
    }

    for cat_col in cond_cat_cols:
        cond_maps[cat_col] = {}
        for num_col in num_cols:
            tmp = ref_df[[cat_col, num_col]].copy()
            tmp[num_col] = pd.to_numeric(tmp[num_col], errors="coerce")
            cond_maps[cat_col][num_col] = tmp.groupby(cat_col, dropna=False)[num_col].median().to_dict()

    return cond_maps, global_medians


def apply_conditional_q50_distance_features(df, cond_maps, global_medians, num_cols, cond_cat_cols):
    df = df.copy()
    for cat_col in cond_cat_cols:
        for num_col in num_cols:
            med_map = cond_maps[cat_col][num_col]
            global_med = global_medians[num_col]

            cond_median = df[cat_col].map(med_map).fillna(global_med).astype(float)
            val = pd.to_numeric(df[num_col], errors="coerce").fillna(global_med).astype(float)

            diff = val - cond_median
            df[f"{num_col}_minus_{cat_col}_q50"] = diff.astype(np.float32)
            df[f"{num_col}_minus_{cat_col}_q50_abs"] = np.abs(diff).astype(np.float32)
    return df


# =========================================================
# Frequency
# =========================================================
def add_frequency_features_global(train_df, test_df, orig_df, cols):
    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in cols:
        ref = pd.concat([
            train_df[col].astype(str),
            test_df[col].astype(str),
            orig_df[col].astype(str)
        ], axis=0)

        freq_map = ref.value_counts(normalize=True, dropna=False).to_dict()

        train_df[f"{col}_freq"] = train_df[col].astype(str).map(freq_map).fillna(0.0).astype(np.float32)
        test_df[f"{col}_freq"] = test_df[col].astype(str).map(freq_map).fillna(0.0).astype(np.float32)

    return train_df, test_df


# =========================================================
# Practical OOF TE
# =========================================================
def make_te_mapping(series, y_binary, smoothing=20):
    tmp = pd.DataFrame({"col": series.astype(str), "y": y_binary})
    global_mean = tmp["y"].mean()
    global_std = tmp["y"].std()
    if pd.isna(global_std):
        global_std = 0.0

    stats = tmp.groupby("col")["y"].agg(["mean", "count", "std"])
    stats["std"] = stats["std"].fillna(0.0)
    stats["mean_smooth"] = (
        stats["count"] * stats["mean"] + smoothing * global_mean
    ) / (stats["count"] + smoothing)

    return stats[["mean_smooth", "std"]], float(global_mean), float(global_std)


def add_oof_te_features_outer(
    train_df,
    valid_df,
    test_df,
    y_train,
    te_cols,
    class_ids,
    n_splits=5,
    seed=42,
    smoothing=20,
):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for col in te_cols:
        train_key_full = train_df[col].astype(str)
        valid_key = valid_df[col].astype(str)
        test_key = test_df[col].astype(str)

        for cls in class_ids:
            y_bin = (np.asarray(y_train) == cls).astype(np.int8)

            oof_mean = np.zeros(len(train_df), dtype=np.float32)
            oof_std = np.zeros(len(train_df), dtype=np.float32)

            for tr_idx, va_idx in skf.split(train_df, y_train):
                tr_x = train_df.iloc[tr_idx]
                va_x = train_df.iloc[va_idx]
                y_tr = y_bin[tr_idx]

                stats, global_mean, global_std = make_te_mapping(tr_x[col], y_tr, smoothing=smoothing)

                va_key = va_x[col].astype(str)
                oof_mean[va_idx] = va_key.map(stats["mean_smooth"]).fillna(global_mean).values.astype(np.float32)
                oof_std[va_idx] = va_key.map(stats["std"]).fillna(global_std).values.astype(np.float32)

            train_df[f"{col}_te_mean_cls{cls}"] = oof_mean
            train_df[f"{col}_te_std_cls{cls}"] = oof_std

            full_stats, full_global_mean, full_global_std = make_te_mapping(train_key_full, y_bin, smoothing=smoothing)

            valid_df[f"{col}_te_mean_cls{cls}"] = valid_key.map(full_stats["mean_smooth"]).fillna(full_global_mean).astype(np.float32)
            valid_df[f"{col}_te_std_cls{cls}"] = valid_key.map(full_stats["std"]).fillna(full_global_std).astype(np.float32)

            test_df[f"{col}_te_mean_cls{cls}"] = test_key.map(full_stats["mean_smooth"]).fillna(full_global_mean).astype(np.float32)
            test_df[f"{col}_te_std_cls{cls}"] = test_key.map(full_stats["std"]).fillna(full_global_std).astype(np.float32)

    return train_df, valid_df, test_df


# =========================================================
# Threshold optimization
# =========================================================
def normalize_rows(arr):
    arr = np.asarray(arr, dtype=np.float64)
    row_sum = arr.sum(axis=1, keepdims=True)
    row_sum = np.where(row_sum <= 0, 1.0, row_sum)
    return arr / row_sum


def apply_class_weights(proba, weights):
    out = np.asarray(proba, dtype=np.float64) * np.asarray(weights, dtype=np.float64)
    out = np.clip(out, EPS, None)
    return normalize_rows(out)


def apply_log_bias(proba, bias):
    logp = np.log(np.clip(np.asarray(proba, dtype=np.float64), EPS, 1.0))
    out = np.exp(logp + np.asarray(bias, dtype=np.float64))
    out = np.clip(out, EPS, None)
    return normalize_rows(out)


def score_from_proba(y_true, proba):
    pred_label = np.argmax(proba, axis=1)
    return balanced_accuracy_score(y_true, pred_label)


def objective_weights(params, y_true, proba):
    adj = apply_class_weights(proba, params)
    return -score_from_proba(y_true, adj)


def objective_log_bias(params, y_true, proba):
    adj = apply_log_bias(proba, params)
    return -score_from_proba(y_true, adj)


# =========================================================
# Load data
# =========================================================
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
orig = pd.read_csv(ORIG_PATH)

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()
orig.columns = orig.columns.str.strip()

le = LabelEncoder()
train[TARGET] = le.fit_transform(train[TARGET])
orig[TARGET] = le.transform(orig[TARGET])

class_ids = sorted(train[TARGET].unique().tolist())
n_classes = len(class_ids)
y = train[TARGET].values

# =========================================================
# Heavy FE outside CV
# =========================================================
X_train_base = add_basic_features(train.drop(columns=[ID_COL, TARGET]).copy())
X_test_base  = add_basic_features(test.drop(columns=[ID_COL]).copy())
orig_X_only  = add_basic_features(orig.drop(columns=[TARGET]).copy())

for df_ in [X_train_base, X_test_base, orig_X_only]:
    for c in df_.columns:
        if df_[c].dtype == "bool":
            df_[c] = df_[c].astype(int)

for c in X_train_base.columns:
    if c not in X_test_base.columns:
        X_test_base[c] = np.nan
    if c not in orig_X_only.columns:
        orig_X_only[c] = np.nan

for c in X_test_base.columns:
    if c not in X_train_base.columns:
        X_train_base[c] = np.nan

X_test_base = X_test_base[X_train_base.columns]
orig_X_only = orig_X_only[X_train_base.columns]

cat_cols = X_train_base.select_dtypes(include=["object"]).columns.tolist()

ORIGINAL_NUM_COLS = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]
num_cols = [c for c in ORIGINAL_NUM_COLS if c in X_train_base.columns]
cond_cat_cols = [c for c in ["Crop_Type", "Season"] if c in X_train_base.columns]

orig_feat_for_stats = orig_X_only.copy()
orig_feat_for_stats[TARGET] = orig[TARGET].values

orig_stats_maps = build_orig_stats_maps(
    orig_df=orig_feat_for_stats,
    cols=cat_cols,
    target_col=TARGET,
    class_names=class_ids,
    smoothing=20,
)
X_train_base = apply_orig_stats_features(X_train_base, orig_stats_maps, cat_cols)
X_test_base  = apply_orig_stats_features(X_test_base, orig_stats_maps, cat_cols)

reference_df = orig_X_only.copy() if USE_ORIG_AS_REFERENCE else pd.concat([X_train_base.copy(), X_test_base.copy()], axis=0)

dist_maps = build_distribution_maps_from_reference(reference_df, num_cols, quantiles=(0.5,))
X_train_base = apply_distribution_position_features(X_train_base, dist_maps, num_cols)
X_test_base  = apply_distribution_position_features(X_test_base, dist_maps, num_cols)

X_train_base = apply_quantile_distance_features(X_train_base, dist_maps, num_cols, quantiles=(0.5,))
X_test_base  = apply_quantile_distance_features(X_test_base, dist_maps, num_cols, quantiles=(0.5,))

cond_maps, global_medians = build_conditional_median_maps_from_reference(reference_df, num_cols, cond_cat_cols)
X_train_base = apply_conditional_q50_distance_features(X_train_base, cond_maps, global_medians, num_cols, cond_cat_cols)
X_test_base  = apply_conditional_q50_distance_features(X_test_base, cond_maps, global_medians, num_cols, cond_cat_cols)

freq_cols = cat_cols + [c for c in X_train_base.columns if c.endswith("_qbin10") or c.endswith("_round1")]
X_train_base, X_test_base = add_frequency_features_global(X_train_base, X_test_base, orig_X_only, freq_cols)

for df_ in [X_train_base, X_test_base]:
    for c in df_.columns:
        if df_[c].dtype == "float64":
            df_[c] = df_[c].astype(np.float32)
        elif df_[c].dtype == "int64":
            df_[c] = df_[c].astype(np.int32)

print("Base feature shape before TE:", X_train_base.shape)

Base feature shape before TE: (630000, 601)


In [3]:
# =========================================================
# Nested CV
# =========================================================
outer_cv = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=SEED)

oof = np.zeros((len(train), n_classes), dtype=np.float32)
pred = np.zeros((len(test), n_classes), dtype=np.float32)

fold_scores = []
best_iterations = []

for fold, (tr_idx, va_idx) in enumerate(outer_cv.split(X_train_base, y)):
    print("=" * 60)
    print(f"OUTER FOLD {fold + 1}/{N_OUTER}")
    print("=" * 60)

    X_tr = X_train_base.iloc[tr_idx].copy()
    X_va = X_train_base.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]
    X_te = X_test_base.copy()

    X_tr, X_va, X_te = add_oof_te_features_outer(
        train_df=X_tr,
        valid_df=X_va,
        test_df=X_te,
        y_train=y_tr,
        te_cols=cat_cols,
        class_ids=class_ids,
        n_splits=N_INNER,
        seed=SEED + fold,
        smoothing=20,
    )

    all_cat_now = X_tr.select_dtypes(include=["object"]).columns.tolist()

    for col in all_cat_now:
        X_tr[col] = X_tr[col].astype(str).fillna("NA")
        X_va[col] = X_va[col].astype(str).fillna("NA")
        X_te[col] = X_te[col].astype(str).fillna("NA")

    X_va = X_va[X_tr.columns]
    X_te = X_te[X_tr.columns]

    cat_feature_indices = [X_tr.columns.get_loc(col) for col in all_cat_now]

    model = CatBoostClassifier(**CAT_PARAMS)
    model.fit(
        X_tr,
        y_tr,
        eval_set=(X_va, y_va),
        cat_features=cat_feature_indices,
        use_best_model=True,
        early_stopping_rounds=200,
        verbose=200,
    )

    va_pred = model.predict_proba(X_va).astype(np.float32)
    te_pred = model.predict_proba(X_te).astype(np.float32)

    oof[va_idx] = va_pred
    pred += te_pred / N_OUTER

    fold_score = balanced_accuracy_score(y_va, np.argmax(va_pred, axis=1))
    fold_scores.append(fold_score)

    best_iter = model.get_best_iteration()
    if best_iter is None or best_iter < 0:
        best_iter = CAT_PARAMS["iterations"]
    best_iterations.append(best_iter)

    print(f"Fold balanced_accuracy = {fold_score:.6f}")
    print(f"Best iteration         = {best_iter}")
    print(f"n_features             = {X_tr.shape[1]}")

    del X_tr, X_va, X_te, model, va_pred, te_pred
    gc.collect()

base_score = balanced_accuracy_score(y, np.argmax(oof, axis=1))

print("\n" + "#" * 60)
print("BASE MODEL (WITH ORIG FEATURES ONLY) RESULTS")
print("#" * 60)
print("Fold scores   :", [round(s, 6) for s in fold_scores])
print("Mean fold     :", round(np.mean(fold_scores), 6))
print("OOF score     :", round(base_score, 6))
print("Avg best_iter :", round(np.mean(best_iterations), 2))

# Save raw model outputs
np.save("oof_cat_origfeat_raw.npy", oof)
np.save("pred_cat_origfeat_raw.npy", pred)

OUTER FOLD 1/10
0:	learn: 0.9849591	test: 0.9851866	best: 0.9851866 (0)	total: 163ms	remaining: 13m 35s
200:	learn: 0.9861751	test: 0.9866200	best: 0.9866200 (198)	total: 11s	remaining: 4m 22s
400:	learn: 0.9866097	test: 0.9867666	best: 0.9867828 (363)	total: 21.8s	remaining: 4m 9s
600:	learn: 0.9867740	test: 0.9868477	best: 0.9868482 (546)	total: 31.3s	remaining: 3m 48s
800:	learn: 0.9868792	test: 0.9869433	best: 0.9869433 (800)	total: 40.4s	remaining: 3m 31s
1000:	learn: 0.9869452	test: 0.9869434	best: 0.9869592 (984)	total: 49.3s	remaining: 3m 17s
bestTest = 0.986959224
bestIteration = 984
Shrink model to first 985 iterations.
Fold balanced_accuracy = 0.969080
Best iteration         = 984
n_features             = 829
OUTER FOLD 2/10
0:	learn: 0.9849944	test: 0.9848297	best: 0.9848297 (0)	total: 80.4ms	remaining: 6m 41s
200:	learn: 0.9862350	test: 0.9863163	best: 0.9863163 (200)	total: 12.7s	remaining: 5m 3s
400:	learn: 0.9866417	test: 0.9866714	best: 0.9866714 (399)	total: 25.2s	rem

In [4]:
# =========================================================
# Threshold optimization on OOF only
# =========================================================
x0_weights = np.ones(n_classes, dtype=np.float64)
res_weights = minimize(
    objective_weights,
    x0=x0_weights,
    args=(y, oof),
    method="Nelder-Mead",
    options={"maxiter": THRESH_MAXITER, "disp": True},
)
best_weights = res_weights.x
oof_weighted = apply_class_weights(oof, best_weights)
pred_weighted = apply_class_weights(pred, best_weights)
score_weighted = score_from_proba(y, oof_weighted)

x0_bias = np.zeros(n_classes, dtype=np.float64)
res_bias = minimize(
    objective_log_bias,
    x0=x0_bias,
    args=(y, oof),
    method="Nelder-Mead",
    options={"maxiter": THRESH_MAXITER, "disp": True},
)
best_bias = res_bias.x
oof_bias = apply_log_bias(oof, best_bias)
pred_bias = apply_log_bias(pred, best_bias)
score_bias = score_from_proba(y, oof_bias)

candidates = [
    {"name": "baseline", "score": base_score, "oof": oof, "pred": pred, "params": None},
    {"name": "class_weight", "score": score_weighted, "oof": oof_weighted, "pred": pred_weighted, "params": best_weights.tolist()},
    {"name": "log_bias", "score": score_bias, "oof": oof_bias, "pred": pred_bias, "params": best_bias.tolist()},
]
best_result = max(candidates, key=lambda x: x["score"])

final_oof = best_result["oof"]
final_pred = best_result["pred"]
final_score = best_result["score"]

print("\n" + "#" * 60)
print("THRESHOLD OPTIMIZATION")
print("#" * 60)
print(f"Base score      : {base_score:.6f}")
print(f"Class weight    : {score_weighted:.6f}")
print(f"Log bias        : {score_bias:.6f}")
print(f"Selected method : {best_result['name']}")
print(f"Final score     : {final_score:.6f}")

# =========================================================
# Save
# =========================================================
np.save("oof_cat_origfeat_threshold_opt.npy", final_oof)
np.save("pred_cat_origfeat_threshold_opt.npy", final_pred)

meta = {
    "base_score": float(base_score),
    "weight_score": float(score_weighted),
    "log_bias_score": float(score_bias),
    "selected_strategy": best_result["name"],
    "selected_score": float(final_score),
    "best_weights": best_weights.tolist(),
    "best_log_bias": best_bias.tolist(),
}
with open("cat_origfeat_threshold_opt_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved: oof_cat_origfeat_raw.npy")
print("Saved: pred_cat_origfeat_raw.npy")
print("Saved: oof_cat_origfeat_threshold_opt.npy")
print("Saved: pred_cat_origfeat_threshold_opt.npy")
print("Saved: cat_origfeat_threshold_opt_meta.json")

# =========================================================
# Submission
# =========================================================
test_label = np.argmax(final_pred, axis=1)
test_label = le.inverse_transform(test_label)

sub = pd.DataFrame({
    "id": test[ID_COL],
    "Irrigation_Need": test_label
})
sub.to_csv("submission.csv", index=False)

print("Saved: submission.csv")
print(sub.head())

Optimization terminated successfully.
         Current function value: -0.975697
         Iterations: 47
         Function evaluations: 102
Optimization terminated successfully.
         Current function value: -0.969762
         Iterations: 3
         Function evaluations: 14

############################################################
THRESHOLD OPTIMIZATION
############################################################
Base score      : 0.969762
Class weight    : 0.975697
Log bias        : 0.969762
Selected method : class_weight
Final score     : 0.975697
Saved: oof_cat_origfeat_raw.npy
Saved: pred_cat_origfeat_raw.npy
Saved: oof_cat_origfeat_threshold_opt.npy
Saved: pred_cat_origfeat_threshold_opt.npy
Saved: cat_origfeat_threshold_opt_meta.json
Saved: submission.csv
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
